# Lesson 23 · Scaling laws: fitting Chinchilla's law ourselves on 75 tiny GPTs, then asking where “20 tokens per parameter” comes from

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/23-quy-luat-mo-rong-en.ipynb)

This notebook accompanies [Lesson 23 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/quy-luat-mo-rong/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/23-quy-luat-mo-rong.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
The loss of a language model falls steadily and predictably as the number of parameters $N$ and the number of
training tokens $D$ grow, but it doesn't fall forever: it has a floor.
We check this ourselves on a small, real sweep: character-level GPTs trained on Vietnamese Wikipedia. We fit
$L(N, D) = E + A/N^\alpha + B/D^\beta$ exactly as Chinchilla did, then use the same formula with the published constants.

**What you'll do:**
1. Count parameters and FLOP: why training costs $C \approx 6ND$, checked on GPT-2 small and Llama 3 8B.
2. Write the Huber loss in log space, its gradient and a BFGS optimizer ourselves, then refit the demo's curve.
3. See the floor $E$, and use the fit to predict a larger model it has never seen.
4. Solve for the compute-optimal frontier (in closed form), compare it with the IsoFLOP valleys, and recompute
   “20 tokens per parameter” from Chinchilla's three sets of constants.
5. Work out when a small model trained for longer (Llama 3 8B) ends up cheaper, and why “emergent” abilities are
   mostly due to the metric.

**Background:** [Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/)
(cross-entropy, nats), [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/)
(the matrices in one block) and
[Lesson 22 · Pretraining & fine-tuning](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/tien-huan-luyen/).
**Run time:** under a minute on a CPU, including the first download of three result files (~70 KB).

Every number in the notebook is computed by the cells below (or read from the result files the demo uses), and
matches the numbers in the demo. Only the 306 training runs (about 10 CPU hours) are not rerun: we load the results
the demo kept, from the very files the demo uses; the notebook refits, extrapolates and computes everything else on
the spot.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.
The function `sci()` prints very large or very small numbers in the form $5.76 \cdot 10^{23}$.

In [ ]:
import hashlib
import itertools
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(v, digits=2):
    """5.76·10²³ · 10¹³ · 2.5·10⁻⁵"""
    e = math.floor(math.log10(abs(v)))
    m = round(v / 10.0**e, digits)
    if abs(m) >= 10:
        m, e = m / 10, e + 1
    head = "" if abs(m - 1) < 1e-12 else f"{fmt(m, digits).rstrip('0').rstrip('.')}·"
    return f"{head}10{str(e).translate(SUP)}"


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is
    long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(fmt(13952, 0), fmt(1.8344, 2), pct(0.0714, 0), sci(5.76e23), sci(1e13))

## 2 · The data: 306 real training runs

The lesson's demo trained **character-level** GPTs (each token is one character, Lesson 14) on 2,185 Vietnamese
Wikipedia articles at least 5,000 characters long (Wikimedia, CC BY-SA). The architecture is Lesson 17's GPT scaled
down: pre-normalization (pre-LN), GELU, learned positions, a context of 64 characters; AdamW with warmup, then a
cosine schedule that decays $\eta$ to 0.1 times its peak exactly after $D$ characters, as Chinchilla did. Each cell of
the grid (model size × number of characters) has its own $\eta$, chosen on the validation set; every loss below is
measured on a separate test set, articles the models never trained on.

Retraining everything would take about 10 CPU hours, so we load the demo's results (`fetch_json`, with an MD5 check):
- `scaling-runs.json`: the loss of every run the lesson uses (the grid, the held-out size, IsoFLOP), the $\eta$
  chosen for each cell (the $\eta$ sweep leaves only that choice), plus the “emergence” measurements and sample
  text;
- `scaling-fit.json`: the fits, the bootstrap and the IsoFLOP parabolas the demo computed (to compare with ours);
- `scaling-chinchilla.json`: Chinchilla's published constants, and the N, D of real models.

In [ ]:
# Results of the long training runs (the same files as the demo on the web). build.py replaces COURSE_DATA_URL with
# the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


RUNS = fetch_json("scaling-runs.json", "897d2bada61f2552e89d15f2631c422a")
FITS = fetch_json("scaling-fit.json", "7a5e470607ddc373654fe32a92385bcd")
CHIN = fetch_json("scaling-chinchilla.json", "35147062a443e4e72c83311d6ccc63e6")
corpus, setup = RUNS["corpus"], RUNS["setup"]
print(
    f"{fmt(corpus['kept'], 0)} articles; training / validation / test characters: "
    + " / ".join(fmt(corpus["chars"][k], 0) for k in ("train", "val", "test"))
    + f"; an alphabet of {corpus['vocab']} characters"
)
print(
    f"the whole sweep: {setup['nRuns']} training runs, {fmt(setup['cpuHours'], 1)} CPU hours; batch {setup['B']} × {setup['T']} characters"
)
print(
    f"only articles ≥ {fmt(corpus['minChars'], 0)} characters long are kept; at the end of the cosine schedule, η is {fmt(0.1, 1)} times its peak"
)

## 3 · Three knobs: $N$, $D$ and $C \approx 6ND$

- $N$: the number of parameters of the model.
- $D$: the number of training tokens, here the number of **characters**.
- $C$: the total number of operations (FLOP) for training. FLOP is a *number* of operations, while FLOP/s is a
  *speed*.

Why is $C \approx 6ND$? Take a linear layer $Y = XW$ with $W$ of size $d_\text{in} \times d_\text{out}$, applied to
$n$ tokens. The forward pass costs $n \cdot d_\text{in} d_\text{out}$ multiply-accumulate (MAC) operations, i.e.
exactly one MAC per weight per token. The backward pass (Lesson 06) needs **two** matrix multiplications of the same
size: $\partial L/\partial X = (\partial L/\partial Y)\,W^\top$ to pass the gradient on to the previous layer, and
$\partial L/\partial W = X^\top (\partial L/\partial Y)$ to update the weights.
Altogether that is 3 MACs = 6 FLOP per weight per token. The cell below counts the MACs of the three multiplications
from their sizes, and checks the gradient with respect to $W$ with finite differences.

In [ ]:
rng = np.random.default_rng(0)
n, d_in, d_out = 64, 48, 32
X, W = rng.normal(size=(n, d_in)), rng.normal(size=(d_in, d_out))
G = rng.normal(size=(n, d_out))  # ∂L/∂Y of a loss L = Σ Y ⊙ G


def macs(A, B):
    """A (p×q)·(q×r) matrix multiplication costs p·q·r multiply-accumulates."""
    return A.shape[0] * A.shape[1] * B.shape[1]


Y = X @ W  # forward pass
dX = G @ W.T  # backward pass: the gradient with respect to the input, passed on to the previous layer
dW = X.T @ G  # backward pass: the gradient with respect to the weights
counts = [macs(X, W), macs(G, W.T), macs(X.T, G)]
print("MACs forward / backward for X / backward for W:", " / ".join(fmt(c, 0) for c in counts))
print(f"= {sum(counts) // (n * W.size)} MACs, i.e. {2 * sum(counts) // (n * W.size)} FLOP, per weight per token")

E_ij = np.zeros_like(W)
E_ij[3, 5] = 1e-6
num = (((X @ (W + E_ij)) * G).sum() - ((X @ (W - E_ij)) * G).sum()) / 2e-6
print(f"∂L/∂W[3, 5]: formula {fmt(dW[3, 5], 6)}, finite differences {fmt(num, 6)}")
assert abs(num - dW[3, 5]) < 1e-6 and sum(counts) == 3 * n * W.size

The whole model is many matrix multiplications like these, so $C \approx 6 \cdot N \cdot D$. The formula leaves out
two things:
- the **attention** part ($q \cdot k$ and the weighted sum of the $v$, Lesson 16), which scales with the context
  length, not with $N$;
- the **embedding-table lookup**: row $x_t$ of the table is *selected*, not multiplied, so the table's parameters
  cost almost no computation.

The cell below counts the parameters from the published configurations of GPT-2 small (Lesson 17) and Llama 3 8B,
then counts every matrix multiplication for one training token. Attention is averaged over positions $1 \dots$ the
context length, because the $t$-th token looks at $t$ tokens. This is also how the demo counts.

In [ ]:
def params(c):
    """Parameter count by part. GPT-2: GELU, LayerNorm (scale + shift), biases, learned positions.
    Llama: SwiGLU (3 matrices), RMSNorm (scale only), RoPE (no position parameters), no biases."""
    L, d, q, kv, ff, V = c["L"], c["d"], c["heads"] * c["hd"], c["kv"] * c["hd"], c["ff"], c["V"]
    gpt2 = c["mlp"] == "GELU"
    attn = L * (d * (q + 2 * kv) + q * d + ((q + 2 * kv + d) if gpt2 else 0))
    mlp = L * ((2 * d * ff + ff + d) if gpt2 else 3 * d * ff)
    norm = (2 * L + 1) * (2 * d if gpt2 else d)
    embed, pos = V * d, (c["ctx"] * d if gpt2 else 0)
    out = 0 if c["tied"] else V * d  # GPT-2 reuses the embedding table (Eᵀ) as its output layer
    total = embed + pos + attn + mlp + norm + out
    return {"total": total, "nonemb": total - embed - pos - out}


def train_flop(c, ctx):
    """FLOP per training token: 6 FLOP for each MAC of every matrix multiplication."""
    L, d, q, kv, ff = c["L"], c["d"], c["heads"] * c["hd"], c["kv"] * c["hd"], c["ff"]
    blocks = L * (d * (q + 2 * kv) + q * d + (2 if c["mlp"] == "GELU" else 3) * d * ff)
    out = d * c["V"]  # output layer: a logit for every token of the vocabulary
    attn = 2 * L * q * (ctx + 1) / 2  # q·kᵀ, then the weighted sum of v, over (ctx + 1)/2 positions on average
    return {"total": 6 * (blocks + out + attn), "attn": 6 * attn}


def config(L, d, heads, kv, hd, ff, V, ctx, mlp, tied):
    """Blocks, width, Q heads, K/V heads, head size, MLP width, vocabulary, context, MLP type, output layer = Eᵀ?"""
    return {"L": L, "d": d, "heads": heads, "kv": kv, "hd": hd, "ff": ff, "V": V, "ctx": ctx, "mlp": mlp, "tied": tied}


# OpenAI's hparams.json (GPT-2 124M) and Llama 3 8B's config.json (Grattafiori et al. 2024, table 3)
GPT2 = config(12, 768, 12, 12, 64, 3072, 50257, 1024, "GELU", tied=True)
LLAMA3_8B = config(32, 4096, 32, 8, 128, 14336, 128256, 8192, "SwiGLU", tied=False)
for name, c in (("GPT-2 small", GPT2), ("Llama 3 8B", LLAMA3_8B)):
    p, f = params(c), train_flop(c, c["ctx"])
    print(
        f"{name}: N = {fmt(p['total'], 0)} (non-embedding {fmt(p['nonemb'], 0)}); context {fmt(c['ctx'], 0)}: "
        f"{fmt(f['total'], 0)} FLOP/token = {fmt(f['total'] / (6 * p['total']), 3)} × 6N, attention {pct(f['attn'] / f['total'], 1)}"
    )
gp, gf = params(GPT2), train_flop(GPT2, 1024)
print(f"GPT-2 small, 6N without the embeddings: {fmt(6 * gp['nonemb'] / 1e6, 0)} million FLOP/token")
TABLE_A4 = [1.03, 1.10, 1.08, 1.04, 1.03, 0.99]  # Hoffmann et al. 2022, table A4: full count / 6ND, 73M…6.8B
print(f"Chinchilla, table A4: full count / 6ND = {fmt(min(TABLE_A4), 2)}–{fmt(max(TABLE_A4), 2)}")
print(
    f"GPT-2 small, full count / 6N = {fmt(gf['total'] / (6 * gp['total']), 2)}; Llama 3 8B's embedding table: "
    f"{fmt(LLAMA3_8B['V'], 0)} × {fmt(LLAMA3_8B['d'], 0)} = {fmt(LLAMA3_8B['V'] * LLAMA3_8B['d'] / 1e6, 0)} million parameters"
)
assert (gp["total"], gp["nonemb"], gf["total"]) == (124439808, 85056000, 797870592), "differs from the demo"
assert params(LLAMA3_8B)["total"] == 8030261248, "differs from the demo"

GPT-2 small has exactly 124,439,808 parameters, like the real model. The full count comes to 797,870,592 FLOP per
token, 1.07 times $6N$. About 7% is attention, and that share grows with the context (12.5% with Llama 3's context
of 8,192). Chinchilla's table A4 also finds the ratio “full count / $6ND$” in the range 0.99–1.10. If, like Kaplan,
you don't count the embedding table, $6N$ is only 510 million, well below the real count, because GPT-2's output
layer *is* the transposed embedding table, and the output layer does do a matrix multiplication.

Llama 3 8B's embedding table has 525 million parameters, plus an output layer of the same size: more than 1 billion
out of 8 billion. How $N$ is counted (with or without the embeddings) comes back in section 10.

## 4 · Measuring it ourselves: loss vs. $D$ for each model size

The demo's grid: 5 model sizes (width $d$ = 16…96) × 5 amounts of data ($D = 2^{18} \dots 2^{22}$ characters) ×
3 seeds. Each run has a learning-rate schedule **exactly as long as** its $D$. Outside the grid there is one larger
size, $d = 128$, which we don't use for fitting: it is the test of extrapolation in section 7.

In [ ]:
size = {s["d"]: s for s in RUNS["sizes"]}
grid = [r for r in RUNS["runs"] if r["kind"] == "grid"]
held = [r for r in RUNS["runs"] if r["kind"] == "held"]
iso_runs = [r for r in RUNS["runs"] if r["kind"] == "iso"]
DS = setup["grid"]
DMAX = DS[-1]
SIZES = sorted({r["d"] for r in grid})
for d, s in size.items():
    tag = "" if d in SIZES else "  ← held out, not used for fitting"
    print(f"d {d:>3}: {s['L']} blocks, N = {fmt(s['total'], 0):>7} (non-embedding {fmt(s['nonemb'], 0):>7}){tag}")
print(f"{len(grid)} runs in the grid, {len(held)} of the held-out size, {len(iso_runs)} for IsoFLOP (section 9)")
print("D (characters):", ", ".join(fmt(D, 0) for D in DS))


def cell(d, D, runs=grid + held):
    return np.array([r["test"] for r in runs if r["d"] == d and r["D"] == D])


assert all(len(cell(d, D)) == 3 for d in SIZES for D in DS)

The figure below plots the test loss (nats per character, mean of 3 seeds; the vertical bars span the 3 seeds)
against $D$, one color and one marker per size, on a log–log scale. A pure power law $L = B/D^\beta$ is a straight
line on this scale.

In [ ]:
MARKS = ["o", "s", "^", "D", "v", "P"]
RAMP = ["#a9c9f2", "#7aaced", "#3987e5", "#2563b0", "#173f73"]  # light → dark: small → large size


def plot_grid(ax, minus=0.0):
    for i, d in enumerate(SIZES):
        m = np.array([cell(d, D).mean() for D in DS]) - minus
        lo = m - (np.array([cell(d, D).min() for D in DS]) - minus)
        hi = (np.array([cell(d, D).max() for D in DS]) - minus) - m
        ax.errorbar(DS, m, yerr=[lo, hi], color=RAMP[i], marker=MARKS[i], ms=6, lw=1.5, capsize=2,
                    label=f"d {d}: {fmt(size[d]['total'], 0)} parameters")  # fmt: skip
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("D (training characters, log scale)")


def pow2_ticks(ax):
    """Ticks 2¹⁸ … 2²² on the D axis; call after fmt_axes."""
    ax.set_xticks(DS, [f"2{str(round(math.log2(D))).translate(SUP)}" for D in DS])
    ax.xaxis.set_minor_formatter(NullFormatter())


fig, ax = plt.subplots(figsize=(8, 4.2))
plot_grid(ax)
ax.set(ylabel="test L (nat/character, log scale)", title="Test loss vs. data, 5 model sizes × 3 seeds")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.3, which="both")
fmt_axes(fig)
ax.set_yticks([1.8, 2.0, 2.2, 2.4, 2.6])  # after fmt_axes: this short log axis has only one 1–2–5 tick
pow2_ticks(ax)
plt.show()
m16 = [cell(16, D).mean() for D in DS]
print("d 16, each doubling of D takes off:", " · ".join(fmt(a - b, 3) for a, b in itertools.pairwise(m16)), "nat")
print(f"d 16 trained on {fmt(DMAX, 0)} characters: L = {fmt(m16[-1], 3)}")
gaps = [np.ptp([cell(d, D).mean() for d in SIZES]) for D in (DS[0], DMAX)]
print(
    f"the smallest and largest sizes are {fmt(gaps[0], 2)} nat apart at D = 2¹⁸ (128 steps), {fmt(gaps[1], 2)} at D = 2²²"
)
spread = [np.ptp(cell(d, D)) for d in SIZES for D in DS]
print(f"the 3 seeds of a cell differ by a median of {fmt(np.median(spread), 3)}, at most {fmt(max(spread), 3)} nat")

Every curve goes down, and larger sizes sit lower, but the smallest model's curve **gradually bends**: the first
doubling of the data takes off 0.159 nat, the last only 0.065. The difference between the 3 seeds of a cell (median
0.014 nat) is much smaller than the gaps between cells, so this shape is real, not noise. At the smallest $D$, only
128 training steps, the sizes are only 0.08 nat apart (0.26 at $2^{22}$): without enough training, a larger model
has not yet had the time to pay off.

## 5 · Fitting $L(N, D) = E + A/N^\alpha + B/D^\beta$ as Chinchilla did

Hoffmann et al. (2022) proposed the form

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

- $E$ is the floor: the loss that remains when both $N$ and $D$ go to infinity.
- $A/N^\alpha$ is the part due to the model not being large enough, $B/D^\beta$ the part due to not having learned
  from enough data.

How to fit it (the paper's third approach, appendix D.2):
- Set $A = e^a$, $B = e^b$, $E = e^e$ so that the three constants always stay positive. Then
  $\ln L = \mathrm{LSE}(a - \alpha \ln N,\ b - \beta \ln D,\ e)$, with $\mathrm{LSE}(u) = \ln \sum_k e^{u_k}$.
  Compute the LSE by subtracting $\max u$ first, so that nothing overflows.
- Compare **in log space**: the residual $r = \ln \hat L - \ln L$ is a relative error.
- Minimize $\sum_i \mathrm{Huber}_\delta(r_i)$ with $\delta = 10^{-3}$: quadratic when $|r| \le \delta$, linear
  beyond, so a run that is far off pulls on the fit only gently.

The cell below writes the objective function together with its gradient with respect to
$\theta = (a, b, e, \alpha, \beta)$, then checks the gradient with finite differences.

In [ ]:
DELTA = 1e-3


def objective(th, lN, lD, lL, delta=DELTA):
    """Σ Huber_δ(LSE(a − α·ln N, b − β·ln D, e) − ln L) and its gradient with respect to θ = (a, b, e, α, β)."""
    a, b, e, al, be = th
    u = np.stack([a - al * lN, b - be * lD, np.full_like(lN, e)])  # the logs of the three terms A/N^α, B/D^β, E
    m = u.max(0)
    w = np.exp(u - m)
    s = w.sum(0)
    r = m + np.log(s) - lL  # numerically stable LSE: ln(prediction) − ln(measured)
    w /= s  # each term's share of the prediction = ∂LSE/∂u
    inside = np.abs(r) <= delta
    f = np.where(inside, 0.5 * r * r, delta * (np.abs(r) - 0.5 * delta)).sum()
    h = np.where(inside, r, delta * np.sign(r))  # Huber′(r)
    g = np.array([(h * w[0]).sum(), (h * w[1]).sum(), (h * w[2]).sum(), -(h * w[0] * lN).sum(), -(h * w[1] * lD).sum()])
    return float(f), g


def points(runs, count="total", sched="sched"):
    """(N, D, L) for each run; N counts every parameter ("total") or leaves out the embeddings ("nonemb")."""
    longest = {(r["d"], r["seed"]): r for r in grid if r["D"] == DMAX}
    out = []
    for r in runs:
        L = r["test"]
        if sched == "inter" and r["D"] < DMAX:  # Kaplan-style (section 10): read partway through the longest run
            L = next(t for ch, _, t in longest[(r["d"], r["seed"])]["inter"] if ch == r["D"])
        out.append((size[r["d"]][count], r["D"], L))
    return np.array(out, float)


P_all = points(grid)
lN, lD, lL = np.log(P_all.T)
th = np.array([5.0, 7.0, 0.4, 0.7, 0.5])
f, g = objective(th, lN, lD, lL)
num = []
for i in range(5):
    step = np.zeros(5)
    step[i] = 1e-6
    num.append((objective(th + step, lN, lD, lL)[0] - objective(th - step, lN, lD, lL)[0]) / 2e-6)
rel = np.abs(np.array(num) - g) / (np.abs(num) + np.abs(g))
print(
    f"{len(P_all)} points; objective at the trial θ = {fmt(f, 5)}; largest relative error of the gradient: {sci(rel.max(), 1)}"
)
assert rel.max() < 1e-5

The gradient matches the finite differences. Now we need an optimizer. Five parameters and a smooth function (except
where Huber's two pieces join): BFGS is the natural choice. It keeps an estimate $H$ of the **inverse** of the
Hessian matrix, moves in the direction $p = -Hg$, chooses the step by backtracking line search (Armijo), then updates
$H$ from the displacement $s$ and the change in gradient $y$ of the step it just took. This is also the optimizer the
demo runs in the browser.

The objective has many local minima, so we start from 32 different points and keep the best result. Chinchilla
used a larger grid; these 32 points are the ones the demo uses when it refits live.

In [ ]:
def bfgs(fn, x0, gtol=1e-13, iters=1000):
    """Minimize fn (which returns the value and the gradient) with BFGS and an Armijo backtracking line search."""
    x = np.array(x0, float)
    f, g = fn(x)
    H = np.eye(len(x))
    for _ in range(iters):
        if np.abs(g).max() < gtol:
            break
        p = -H @ g
        if not p @ g < 0:  # H is no longer positive definite: fall back to the steepest-descent direction
            H, p = np.eye(len(x)), -g
        t = 1.0
        for _ in range(60):
            f_new, g_new = fn(x + t * p)
            if np.isfinite(f_new) and f_new <= f + 1e-4 * t * (p @ g):
                break
            t *= 0.5
        else:
            break  # no step found that decreases f
        s, y = t * p, g_new - g
        x, f, g = x + s, f_new, g_new
        sy = s @ y
        if sy > 1e-300:
            Hy = H @ y
            H += (sy + y @ Hy) * np.outer(s, s) / sy**2 - (np.outer(Hy, s) + np.outer(s, Hy)) / sy
        if np.abs(s).max() < 1e-15:
            break
    return x, f


STARTS = list(itertools.product([0, 10], [5, 15], [-0.5, 0.5], [0.3, 1.0], [0.3, 1.0]))  # (a, b, e, α, β)


def fit(P, scale=1.0, gtol=1e-13, starts=STARTS, delta=DELTA):
    """Fit E + A/N^α + B/D^β to the points (N, D, L); scale = 1/len(P) takes the mean instead of the sum (section 14)."""
    lN, lD, lL = np.log(P.T)

    def fn(th):
        f, g = objective(th, lN, lD, lL, delta)
        return scale * f, scale * g

    th, f = min((bfgs(fn, x0, gtol) for x0 in starts), key=lambda r: r[1])
    a, b, e, al, be = th
    return {"E": np.exp(e), "A": np.exp(a), "B": np.exp(b), "alpha": al, "beta": be, "obj": f / scale, "theta": th}


def predict(f, N, D):
    return f["E"] + f["A"] / np.asarray(N, float) ** f["alpha"] + f["B"] / np.asarray(D, float) ** f["beta"]


def show(f):
    return " · ".join(
        f"{k} = {fmt(f[k], 4 if k in ('alpha', 'beta', 'E') else 1)}" for k in ("E", "A", "B", "alpha", "beta")
    )


f0 = fit(P_all)
stored = FITS["fits"]["all-total-sched"]
print("our fit:   ", show(f0).replace("alpha", "α").replace("beta", "β"))
print("demo saved:", show(stored).replace("alpha", "α").replace("beta", "β"))
res = predict(f0, P_all[:, 0], P_all[:, 1]) - P_all[:, 2]
inside = np.abs(np.log(predict(f0, P_all[:, 0], P_all[:, 1]) / P_all[:, 2])) <= DELTA
print(
    f"residuals: root mean square {fmt(np.sqrt(np.mean(res**2)), 3)} nat; {inside.sum()}/{len(res)} points within |r| ≤ δ"
)
assert f0["obj"] <= stored["obj"] * (1 + 1e-5), "differs from the demo"  # the same minimum as the demo's scipy
assert all(abs(f0[k] - stored[k]) < 2e-3 for k in ("E", "alpha", "beta")), "differs from the demo"
assert show(f0) == show(stored), "differs from the demo"  # every printed digit, A and B included

Our refit gives exactly the demo's curve (identical in every printed digit; $A$ and $B$ differ only in later digits,
because the two optimizers stop at two very close points). Five constants summarize all 75 runs, with an average deviation (root mean square) of
0.035 nat. Only about 1/7 of the points fall in Huber's quadratic region $|r| \le \delta$, so most points count by
$|r|$: with a $\delta = 10^{-3}$ this small, this is almost a fit by the absolute value of the relative error.

What $\alpha$ means: doubling $N$ multiplies $A/N^\alpha$ by $2^{-\alpha}$. Note that this is only the *reducible*
part. Kaplan et al. (2020) fitted $L(N) = (N_c/N)^{\alpha_N}$ to the whole loss, with no floor, and measured
$\alpha_N \approx 0.076$: doubling $N$ multiplies all of $L$ by $2^{-0.076} \approx 0.95$. The two exponents belong to
two different forms of the formula, so they can't be compared directly.

In [ ]:
print(f"2^(−α) = {fmt(2 ** -f0['alpha'], 3)}: doubling N leaves {pct(2 ** -f0['alpha'], 0)} of the A/N^α part")
print(f"2^(−β) = {fmt(2 ** -f0['beta'], 3)}: doubling D leaves {pct(2 ** -f0['beta'], 0)} of the B/D^β part")
print(f"Kaplan: 2^(−0.076) = {fmt(2**-0.076, 3)}")

## 6 · There is a floor: only after subtracting $E$ is it a straight line

On the log–log scale of section 4, the curves bend because the loss cannot go below
$E + A/N^\alpha$, the floor of a model size trained on infinite data. If Chinchilla's form is right, then after
subtracting both $E$ and the $N$ part, every run must lie on **the same straight line** $B/D^\beta$, with slope
$-\beta$, whatever the model size.

In the figure below, left: the measured $L$ and the floor $E + A/N^\alpha$ of each size (horizontal dotted line).
Right: $L - E - A/N^\alpha$ for the same cells (mean of 3 seeds), one marker per size, with the line $B/D^\beta$
(black).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
plot_grid(a1)
for i, d in enumerate(SIZES):
    a1.axhline(f0["E"] + f0["A"] / size[d]["total"] ** f0["alpha"], color=RAMP[i], ls=":", lw=1.2)
a1.set(ylabel="test L (nat/character)", title="L and the floor E + A/N^α of each size")
a1.legend(frameon=False, fontsize=8)
for i, d in enumerate(SIZES):
    m = np.array([cell(d, D).mean() for D in DS])
    a2.plot(DS, m - f0["E"] - f0["A"] / size[d]["total"] ** f0["alpha"], MARKS[i], color=RAMP[i], ms=7, mec="white")
Dl = np.geomspace(DS[0] / 1.3, DS[-1] * 1.3, 50)
a2.plot(Dl, f0["B"] / Dl ** f0["beta"], color="black", lw=1.2, label=f"B/D^β, β = {fmt(f0['beta'], 2)}")
a2.set(xscale="log", yscale="log", xlabel="D (characters, log scale)", ylabel="L − E − A/N^α (nat/character)")
a2.set_title("Floor subtracted: all sizes on one line")
a2.legend(frameon=False)
for ax in (a1, a2):
    ax.grid(alpha=0.3, which="both")
plt.tight_layout()
fmt_axes(fig)
a1.set_yticks([1.6, 1.8, 2.0, 2.2, 2.4, 2.6])
pow2_ticks(a1)
pow2_ticks(a2)
plt.show()
floor16 = f0["E"] + f0["A"] / size[16]["total"] ** f0["alpha"]
print(f"E = {fmt(f0['E'], 3)} nat/character = {fmt(f0['E'] / math.log(2), 2)} bits/character")
print(
    f"the floor of d 16 (13,952 parameters): {fmt(floor16, 3)}; it is at {fmt(m16[-1], 3)}, still {fmt(m16[-1] - floor16, 3)} above it"
)
assert abs(floor16 - 1.834) < 2e-3, "differs from the demo"

On the right, the 25 cells of the 5 model sizes lie almost on one straight line, deviating most at the two ends
(especially the shortest runs, 128 steps): the data part is a genuine power law, the same for every size. On the
left, the smallest model has a floor of about 1.83 nat/character. However much data is added, it cannot get below
that, because the limit lies in the capacity of 13,952 parameters.

And what about $E$? It is the floor of **this fit, on this data, with this kind of model**, not “the entropy of
Vietnamese”. Section 7 will show $E$ changing by up to 0.15 when we change only the set of runs used for fitting, and
section 10 shows it dropping to 1.44 when we change only how the loss is read.

## 7 · Extrapolation: predicting a model never seen

The real value of a scaling law is **prediction**: fit on small, cheap models, then predict a large model before
training it. The held-out model ($d = 128$, 850,880 parameters) is 1.7 times as large as the largest size in the fit
($d = 96$), and has 3 seeds at $D = 2^{21}$ and $2^{22}$ characters.

### 🤔 Predict first
The fit predicts the loss of the held-out model at $D = 2^{22}$. How far off is it: under 0.05 nat, about 0.2 nat,
or more than 0.5 nat?

In [ ]:
def signed(x, digits=3):
    return ("+" if x > 0 else "") + fmt(x, digits)


NH = size[128]["total"]
print(
    f"held-out model: {fmt(NH, 0)} parameters = {fmt(NH / size[SIZES[-1]]['total'], 1)} times the largest size in the fit"
)
for D in (DS[-2], DMAX):
    meas, p = cell(128, D), predict(f0, NH, D)
    print(
        f"  D = {fmt(D, 0)}: predicted {fmt(p, 3)}, measured {fmt(meas.min(), 3)}–{fmt(meas.max(), 3)} across 3 seeds (mean {fmt(meas.mean(), 3)}); "
        f"off by {signed(p - meas.mean())} nat = {pct(abs(p - meas.mean()) / meas.mean(), 1)} of the loss; "
        f"spread across seeds {fmt(np.ptp(meas), 3)}"
    )
err_held = predict(f0, NH, DMAX) - cell(128, DMAX).mean()
assert abs(err_held - 0.0174) < 2e-3, "differs from the demo"

The fit has never seen this model, yet it is off by less than 0.02 nat:
- At $2^{21}$ characters, it is off by +0.002, within the spread across seeds.
- At $2^{22}$ characters, it is off by +0.017. That is larger than the spread across the 3 seeds (0.004), but only 1%
  of the loss.

The fit is slightly pessimistic (the real model does better than predicted), and that is the safer direction to be
wrong in when deciding whether to train a large model.

One correct prediction doesn't say much. The cell below refits on four subsets of the grid, each time hiding one
part, then predicts the hidden part and the held-out model.

In [ ]:
PRESETS = {
    "all runs with d ≤ 96": ("all", lambda r: True),
    "only D ≤ 2.1 million characters": ("smallD", lambda r: r["D"] <= DS[-2]),
    "only d ≤ 48": ("smallN", lambda r: r["d"] <= 48),
    "without the runs of ≤ 256 steps": ("long", lambda r: r["steps"] > 256),
}
sub = {}
for name, (key, keep) in PRESETS.items():
    f = f0 if key == "all" else fit(points([r for r in grid if keep(r)]))
    sub[key] = f
    hidden = [r for r in grid if not keep(r)]
    line = f"{name:<31}: E {fmt(f['E'], 3)} · α {fmt(f['alpha'], 2)} · β {fmt(f['beta'], 2)} → d 128 at 2²²: {fmt(predict(f, NH, DMAX), 3)}"
    if hidden:
        e = [predict(f, size[r["d"]]["total"], r["D"]) - r["test"] for r in hidden]
        w = int(np.argmax(np.abs(e)))
        line += f"; {len(hidden)} hidden runs, largest error {signed(e[w])} (d {hidden[w]['d']}, {fmt(hidden[w]['steps'], 0)} steps)"
    print(line)
    st = FITS["fits"][f"{key}-total-sched"]
    assert abs(f["alpha"] - st["alpha"]) < 2e-3 and abs(f["beta"] - st["beta"]) < 2e-3, "differs from the demo"
Es = [f["E"] for f in sub.values()]
heldp = [predict(f, NH, DMAX) for f in sub.values()]
print(
    f"E: {fmt(min(Es), 2)}–{fmt(max(Es), 2)} (a spread of {fmt(max(Es) - min(Es), 2)}); predictions for d 128: a spread of {fmt(np.ptp(heldp), 3)} nat"
)

The four fits give constants that are far apart: $E$ differs by 0.15, and $\beta$ ranges from 0.47 to 0.77. Yet all
four predict the held-out model within a range only 0.024 nat wide. **Predictions are far more robust than
exponents.** A few dozen runs spanning two orders of magnitude are enough to predict near the measured region, but not
enough to pin down a “law” with a precise exponent.

The worst error (+0.30 nat) is the 128-step run of the smallest model, when we fit only on the long runs. The form
$E + A/N^\alpha + B/D^\beta$ cannot describe training runs that are too short, where warmup and the learning-rate
schedule take up almost all of the time. Besiroglu et al. (2024) also dropped five anomalous Chinchilla runs (under
0.4 tokens per parameter, with unusually high losses) from their refit.

## 8 · The compute-optimal frontier: a closed-form solution

Now for the question Chinchilla asked: with a fixed budget $C = 6ND$, how should it be split between $N$ and $D$?
Substituting $D = C/(6N)$ into the fit gives a function of one variable

$$L(N) = E + A N^{-\alpha} + B \left(\tfrac{C}{6}\right)^{-\beta} N^{\beta}.$$

Setting the derivative to 0: $\alpha A N^{-\alpha-1} = \beta B (C/6)^{-\beta} N^{\beta-1}$, hence

$$N_\text{opt} = G \left(\tfrac{C}{6}\right)^{\beta/(\alpha+\beta)}, \qquad D_\text{opt} = G^{-1} \left(\tfrac{C}{6}\right)^{\alpha/(\alpha+\beta)}, \qquad G = \left(\tfrac{\alpha A}{\beta B}\right)^{1/(\alpha+\beta)}.$$

We check the solution by scanning 400,001 values of $N$ along the curve of constant budget.

In [ ]:
def frontier(f, C, k=6.0):
    """The compute-optimal N, D and the predicted loss at a budget C = k·N·D."""
    al, be = f["alpha"], f["beta"]
    G = (al * f["A"] / (be * f["B"])) ** (1 / (al + be))
    N = G * (C / k) ** (be / (al + be))
    D = (C / k) ** (al / (al + be)) / G
    return N, D, float(predict(f, N, D))


def brute(f, C, N_guess, k=6.0):
    Ns = np.exp(np.linspace(np.log(N_guess) - 6, np.log(N_guess) + 6, 400001))
    L = predict(f, Ns, C / (k * Ns))
    return Ns[L.argmin()], L.min()


BUDGETS = sorted({r["C"] for r in iso_runs})
for C in BUDGETS:
    N, D, L = frontier(f0, C)
    Nb, Lb = brute(f0, C, N)
    print(
        f"C = {sci(C, 1)}: N_opt = {fmt(N, 0)}, D_opt = {fmt(D, 0)}, D/N = {fmt(D / N, 0)}, L = {fmt(L, 3)}  (scan: N = {fmt(Nb, 0)})"
    )
    assert abs(math.log(Nb / N)) < 1e-3 and Lb >= L - 1e-12
print(f"(each scan covers {fmt(400001, 0)} values of N, evenly spaced in ln N around the solution)")
a_exp = f0["beta"] / (f0["alpha"] + f0["beta"])
print(f"a = β/(α+β) = {fmt(a_exp, 3)}: N_opt ∝ C^{fmt(a_exp, 2)}, D_opt ∝ C^{fmt(1 - a_exp, 2)}")
print(
    "optimal D/N for C = 10¹¹ … 10¹⁴:",
    " → ".join(fmt(frontier(f0, C)[1] / frontier(f0, C)[0], 0) for C in 10.0 ** np.arange(11, 15)),
)

The closed-form solution matches the scan. Two things to notice:
- If $\alpha = \beta$ then $a = 1/2$: $N$ and $D$ grow in step, each time $C$ quadruples both of them double, and the
  ratio $D/N$ stays the same. That is Chinchilla's main conclusion.
- Our fit has $\alpha > \beta$, so $a < 1/2$: the optimal ratio $D/N$ **grows** with $C$, from about 120 to about 320
  characters per parameter as $C$ grows by three orders of magnitude. There is no single fixed ratio.

## 9 · IsoFLOP valleys: measured directly, no fit needed

Chinchilla's second approach needs no formula at all: fix $C$, train several model sizes, each on exactly
$D = C/(6N)$ characters. Too small means too little capacity, too large means too little data, so the loss as a
function of $\ln N$ forms a valley. Fit a parabola in $\ln N$ and read off its bottom. The demo ran 3 such budgets,
each with 4–5 sizes × 3 seeds. The cell below reads off the bottom of each valley, and compares the ratio $D/N$ there
with the ratio the fit of section 8 gives at the same budget.

In [ ]:
def parabola(lnN, L):
    """L ≈ c₂(ln N)² + c₁ ln N + c₀; returns N at the bottom and the loss at the bottom (Chinchilla's approach 2)."""
    c2, c1, c0 = np.polyfit(lnN, L, 2)
    return math.exp(-c1 / (2 * c2)), c0 - c1 * c1 / (4 * c2), (c2, c1, c0)


iso = {}
for C in BUDGETS:
    rs = [r for r in iso_runs if r["C"] == C]
    lnN = np.log([size[r["d"]]["total"] for r in rs])
    L = np.array([r["test"] for r in rs])
    N_par, L_par, coef = parabola(lnN, L)
    seeds = [C / 6 / parabola(lnN[s], L[s])[0] ** 2 for s in [np.array([r["seed"] == k for r in rs]) for k in range(3)]]
    iso[C] = {"runs": rs, "N": N_par, "L": L_par, "coef": coef, "seeds": seeds}
    N_fit, D_fit, _ = frontier(f0, C)
    print(
        f"C = {sci(C, 1)}: {len({r['d'] for r in rs})} sizes × 3 seeds; parabola: bottom at N ≈ {fmt(N_par, 0)}, "
        f"D/N ≈ {fmt(C / 6 / N_par**2, 0)} (per seed {fmt(min(seeds), 0)}–{fmt(max(seeds), 0)}); "
        f"fit: N ≈ {fmt(N_fit, 0)}, D/N ≈ {fmt(D_fit / N_fit, 0)}"
    )
    assert abs(N_par / FITS["iso"][f"{C:g}"]["parabola"]["N"] - 1) < 1e-6, "differs from the demo"
par_ratio = [C / 6 / iso[C]["N"] ** 2 for C in BUDGETS]
fit_ratio = [frontier(f0, C)[1] / frontier(f0, C)[0] for C in BUDGETS]
print(
    "D/N at the bottom, parabola:",
    " → ".join(fmt(v, 0) for v in par_ratio),
    "· fit:",
    " → ".join(fmt(v, 0) for v in fit_ratio),
)
assert 80 < min(par_ratio) and max(par_ratio) < 120 and 135 < min(fit_ratio) and max(fit_ratio) < 215, (
    "differs from the demo"
)

The figure below plots the three valleys: one color and one marker per budget (mean of 3 seeds; the vertical bars
span the 3 seeds). The solid line is the parabola in $\ln N$; the dashed line is the valley that the fit of section 5
predicts, $L(N, C/(6N))$; the vertical line marks the bottom of the parabola.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.3))
for C, color, mk in zip(BUDGETS, (BLUE, ORANGE, GREEN), ("o", "s", "^")):
    v = iso[C]
    ds = sorted({r["d"] for r in v["runs"]})
    Ns = np.array([size[d]["total"] for d in ds])
    vals = [np.array([r["test"] for r in v["runs"] if r["d"] == d]) for d in ds]
    m = np.array([x.mean() for x in vals])
    yerr = [m - [x.min() for x in vals], [x.max() for x in vals] - m]
    ax.errorbar(Ns, m, yerr=yerr, fmt=mk, color=color, ms=7, mec="white", capsize=2, label=f"C = {sci(C, 1)} FLOP")
    xs = np.geomspace(Ns[0] / 1.5, Ns[-1] * 1.5, 100)
    c2, c1, c0 = v["coef"]
    ax.plot(xs, c2 * np.log(xs) ** 2 + c1 * np.log(xs) + c0, color=color, lw=1.4)
    ax.plot(xs, predict(f0, xs, C / (6 * xs)), color=color, lw=1.2, ls="--")
    ax.axvline(v["N"], color=color, lw=0.8, ls=":")
ax.set(xscale="log", xlabel="N (parameters, log scale)", ylabel="test L (nat/character)", ylim=(1.75, 2.6))
ax.set_title("IsoFLOP valleys: solid = parabola in ln N, dashed = fit")
ax.legend(frameon=False)
ax.grid(alpha=0.3, which="both")
fmt_axes(fig)
plt.show()

The valleys are real: at the two larger budgets, a size in the middle beats both ends, and the bottom shifts right as
$C$ grows. At the smallest budget, d 16 and d 32 are tied within the spread across seeds, so the bottom sits right at
the left edge of the sizes measured: that is why the per-seed parabolas here range from 93 to 166. And the two ways
of reading the bottom **don't even agree on the direction**:
- the parabola gives ratios of about 115 → 93 → 85 characters per parameter, i.e. *falling* as $C$ grows;
- the fit gives 139 → 169 → 207, i.e. *rising*.

With 4–5 points per valley, the bottom of the parabola is very sensitive to the points at the two ends, while the fit
is constrained by the form of the formula. Neither is “wrong”: the data just isn't enough to choose between them.
That is why the demo says “about 85–115 characters per parameter according to the IsoFLOP valleys, and 140–210
according to the fit”, rather than a single number.

These are **characters**, not BPE tokens. A GPT-2 token is a few characters long on average (Lesson 14), so this
figure of about 100 doesn't contradict Chinchilla's “20 tokens per parameter” at all. The optimal ratio depends on the
data, the tokenization, the architecture, and even the estimation method.

## 10 · Why did Kaplan recommend growing the model faster?

Kaplan et al. (2020) concluded that $N_\text{opt} \propto C^{0.73}$: most of the budget should go to the model.
Chinchilla found $a \approx 0.5$. Two explanations are usually given:
1. **The learning-rate schedule** (Hoffmann et al.): Kaplan read the loss at a $D$ *partway through* a long run, whose
   learning-rate schedule was longer than $D$, so that loss is higher than that of a run whose schedule ends exactly
   at $D$.
2. **How $N$ is counted** (Pearce & Song, 2024): Kaplan didn't count the embedding parameters. In small models the
   embeddings make up a large share of the parameters, so leaving them out skews the exponent.

The demo's longest runs were also measured partway through ($2^{18} \dots 2^{21}$ characters), so we can test both
ideas on the same runs.

In [ ]:
P_nonemb = points(grid, count="nonemb")
P_inter = points(grid, sched="inter")
diff = P_inter[:, 2] - P_all[:, 2]
part = P_all[:, 1] < DMAX
print(
    f"read partway through vs. a schedule that ends at D: higher by {fmt(diff[part].mean(), 3)} on average, "
    f"by at most {fmt(diff[part].max(), 3)} nat ({part.sum()} runs)"
)
fits_k = {"all parameters, schedule ends at D": f0}
fits_k["without embeddings (Kaplan)"] = fit(P_nonemb)
fits_k["read partway through (Kaplan)"] = fit(P_inter)
for name, f in fits_k.items():
    a_k = f["beta"] / (f["alpha"] + f["beta"])
    print(f"{name:<34}: E {fmt(f['E'], 2)} · α {fmt(f['alpha'], 2)} · β {fmt(f['beta'], 2)} → a = {fmt(a_k, 2)}")
for key, f in (
    ("all-nonemb-sched", fits_k["without embeddings (Kaplan)"]),
    ("all-total-inter", fits_k["read partway through (Kaplan)"]),
):
    st = FITS["fits"][key]
    assert abs(f["alpha"] - st["alpha"]) < 2e-3 and abs(f["beta"] - st["beta"]) < 2e-3, "differs from the demo"
share = {d: 1 - size[d]["nonemb"] / size[d]["total"] for d in (16, 128)}
print(
    f"embeddings + output layer + positions: {pct(share[16], 0)} of the parameters of d 16, {pct(share[128], 0)} of d 128"
)

- **Not counting the embeddings** pushes $a$ from 0.43 up to 0.50, i.e. toward “grow the model faster”, the same
  direction as Kaplan. The reason: the character embeddings, the positions and the output layer make up 53% of the
  parameters of the smallest model but only 7% of d 128, so leaving them out makes the small models “smaller” by more
  than the large ones.
- **Reading partway through** is higher by 0.047 nat on average, just as Hoffmann says, but here it pushes $a$ the
  *opposite* way (0.30), and pulls $E$ down to 1.44. The learning-rate schedule cannot explain the Kaplan–Chinchilla
  difference in this sweep. Porian et al. (2024) also found that careful learning-rate decay is “not essential”.
  According to them, the difference comes from three other things: the FLOP of the output layer being ignored, a
  warmup that is too long for small models, and not tuning the optimizer for each scale.

The general lesson: the exponents of a scaling law are sensitive to choices that look like technical details (what is
counted, when the loss is read, how $\eta$ is tuned).

## 11 · Where does “20 tokens per parameter” come from?

Chinchilla (70 billion parameters, 1.4 trillion tokens) was trained on exactly 20 tokens per parameter, and the number
20 became a rule of thumb. The paper has three estimation approaches. Approaches 1 and 2 give $a \approx 0.5$, while
approach 3 is a fit like the one we just did, and prints the constants $E = 1.69$, $A = 406.4$, $B = 410.7$,
$\alpha = 0.34$, $\beta = 0.28$. The paper's TeX source has more precise values. Besiroglu et al. (2024) reconstructed
the data from the paper's figures and refitted it.

### 🤔 Predict first
At Gopher's budget, $C = 5.76 \cdot 10^{23}$ FLOP, what optimal number of tokens per parameter do the very constants
printed in the paper give: about 20, about 50, or about 90?

In [ ]:
CH_FITS = {  # Hoffmann et al. 2022, appendix D.2; the paper's TeX source; Besiroglu et al. 2024, equation 4
    "printed in paper": {"E": 1.69, "A": 406.4, "B": 410.7, "alpha": 0.34, "beta": 0.28},
    "TeX source": {"E": 1.6934, "A": 406.4, "B": 410.7, "alpha": 0.3392, "beta": 0.2849},
    "Besiroglu 2024": {"E": 1.8172, "A": 482.01, "B": 2085.43, "alpha": 0.3478, "beta": 0.3658},
}
assert [CH_FITS[k] for k in CH_FITS] == [CHIN["fits"][k] for k in ("paper", "tex", "refit")]  # the demo's constants
C_GOPHER = 5.76e23
ratio_gopher = {}
for name, f in CH_FITS.items():
    N, D, L = frontier(f, C_GOPHER)
    Nb, _ = brute(f, C_GOPHER, N)
    assert abs(math.log(Nb / N)) < 1e-3
    ratio_gopher[name] = D / N
    other = " · ".join(f"{sci(C, 0)}: {fmt(frontier(f, C)[1] / frontier(f, C)[0], 0)}" for C in (1e21, 1e25, 1e27))
    print(
        f"{name:<16}: α = {fmt(f['alpha'], 4)}, β = {fmt(f['beta'], 4)}, a = {fmt(f['beta'] / (f['alpha'] + f['beta']), 3)}; at Gopher's C, N_opt = {fmt(N / 1e9, 1)} billion, "
        f"D_opt = {fmt(D / 1e12, 2)} trillion, D/N = {fmt(D / N, 1)}  (at {other})"
    )
assert abs(ratio_gopher["printed in paper"] - 92.6) < 0.1 and abs(ratio_gopher["Besiroglu 2024"] - 18.4) < 0.1, (
    "differs from the demo"
)

The very constants printed in the paper give about **93** tokens per parameter at Gopher's budget, nearly five times
the 20 that Chinchilla was trained on. Just using the more precise exponents of the TeX source ($\alpha$ 0.3392
instead of 0.34, $\beta$ 0.2849 instead of 0.28) already brings the ratio down to 59: the ratio is sensitive to the
second and third digits of the exponents. And because $\alpha \ne \beta$, the ratio also changes with $C$.

The refit by Besiroglu et al. gives about **18**, and changes slowly with $C$ (22 at $10^{21}$, 15 at $10^{27}$)
because $\alpha \approx \beta$. They point out that the original took the *mean* of the Huber loss instead of the
*sum*, so the optimizer stopped too early (section 14 tries this again). They also argue that the paper's confidence
intervals are implausibly narrow. “20 tokens per parameter” agrees with approach 1, approach 2 and the refit, but it
is a rule of thumb, not a constant of nature.

The figure below plots the optimal ratio against the budget for the three sets of constants (three line styles),
together with real models placed at their own $C = 6ND$ (orange diamonds, labeled).

In [ ]:
models = {m["name"]: m for m in CHIN["models"]}
Cs = np.geomspace(1e19, 1e27, 60)
fig, ax = plt.subplots(figsize=(8.5, 4.4))
for (name, f), ls in zip(CH_FITS.items(), ("-", "--", ":")):
    ax.plot(Cs, [frontier(f, C)[1] / frontier(f, C)[0] for C in Cs], color=BLUE, ls=ls, lw=2, label=name)
OFFSET = {"GPT-3": (-36, 6), "Gopher": (8, -4), "Llama 3 8B": (8, -4), "Llama 3.1 405B": (-40, -14)}
for name, m in models.items():
    ax.plot(6 * m["N"] * m["D"], m["D"] / m["N"], "D", color=ORANGE, ms=7, mec="white")
    xy = (6 * m["N"] * m["D"], m["D"] / m["N"])
    ax.annotate(name, xy, xytext=OFFSET.get(name, (6, 4)), textcoords="offset points", fontsize=9)
ax.axhline(20, color=GREY, lw=1, ls="-.")
ax.text(2e26, 11, "20 tokens/parameter", color=GREY, fontsize=9, ha="center")
ax.set(
    xscale="log",
    yscale="log",
    xlabel="C (training FLOP, log scale)",
    ylabel="D/N (tokens per parameter)",
    ylim=(0.5, 4000),
)
ax.set_title("Compute-optimal ratio under Chinchilla's three sets of constants")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, 0.9))
ax.grid(alpha=0.3, which="both")
fmt_axes(fig)
plt.show()
for name in ("GPT-3", "Gopher", "Chinchilla", "Llama 3 8B", "Llama 3 70B", "Llama 3.1 405B"):
    m = models[name]
    print(
        f"{name:<15}: N = {fmt(m['N'] / 1e9, 1)} billion, D = {fmt(m['D'] / 1e12, 2)} trillion, D/N = {fmt(m['D'] / m['N'], 1)}"
    )
above = {}  # each Llama's D/N against the three lines, at its own C = 6ND
for name in ("Llama 3 8B", "Llama 3 70B", "Llama 3.1 405B"):
    m = models[name]
    c_own = 6 * m["N"] * m["D"]
    above[name] = [m["D"] / m["N"] > frontier(f, c_own)[1] / frontier(f, c_own)[0] for f in CH_FITS.values()]
assert above == {"Llama 3 8B": [True] * 3, "Llama 3 70B": [True] * 3, "Llama 3.1 405B": [False, False, True]}, (
    "differs from the text below"
)

There are three things worth noticing in the figure (the curves are the optimal ratio under each set of constants,
the orange diamonds are real models):
- Gopher and GPT-3 sit very low, at about 1–2 tokens per parameter: too large for their data, on the “too little
  data” side of the valley.
- Chinchilla sits right on the refit's line.
- Llama 3 8B and 70B sit *above* every line, Llama 3 8B at nearly 1,900 tokens per parameter; Llama 3.1 405B (38.5)
  sits above the refit's line only. That is not a mistake: the next section works out why.

Llama 3 also fitted its own scaling law on its own data, and chose 402 billion parameters for 16.55 trillion tokens
at $3.8 \cdot 10^{25}$ FLOP. The cell below computes that ratio.

In [ ]:
l3 = CHIN["llama3Fit"]
print(
    f"Llama 3, its own fit: {fmt(l3['N'] / 1e9, 0)} billion parameters, {fmt(l3['D'] / 1e12, 2)} trillion tokens at "
    f"{sci(l3['C'], 1)} FLOP → D/N ≈ {fmt(l3['D'] / l3['N'], 0)}; 6ND = {sci(6 * l3['N'] * l3['D'], 1)}"
)
N_r, D_r, _ = frontier(CH_FITS["Besiroglu 2024"], l3["C"])
print(f"Chinchilla's refit at the same budget: {fmt(N_r / 1e9, 0)} billion parameters, D/N ≈ {fmt(D_r / N_r, 0)}")

Llama 3 did not fit the form $E + A/N^\alpha + B/D^\beta$: they measured IsoFLOP valleys (like Chinchilla's
approach 2, section 9) on their own data. Different data and tokenizer: 41 instead of 17. (That pair of numbers gives
$6ND = 4 \cdot 10^{25}$, slightly more than the $3.8 \cdot 10^{25}$ budget the paper states, so their FLOP count is
not exactly $6ND$.) The optimal ratio is a property of *the data, the tokenization and the architecture*, not of
“language models” in general.

## 12 · Optimal for training is not necessarily optimal for use

The compute-optimal frontier answers the question “what is cheapest to **train**”. But a model is used to generate
text, sometimes trillions of tokens, and each generated token costs about $2N$ FLOP (forward pass only, ignoring
attention and the KV cache, Lesson 17). With $T$ tokens to be generated, the total cost is

$$\text{cost}(N) = 6 N D(N) + 2 N T,$$

where $D(N)$ is the number of tokens a model with $N$ parameters needs to reach a target loss (solved from the fit:
$D = \big(B / (L - E - A/N^\alpha)\big)^{1/\beta}$). Sardana et al. (2024) propose minimizing exactly this sum.

### 🤔 Predict first
According to Chinchilla's refit, the loss level of Llama 3 8B (15 trillion tokens) could be reached by a larger
compute-optimal model trained on fewer tokens. About how large is that model, and how many tokens have to be
generated before the 8B model “breaks even”?

In [ ]:
def tokens_for(f, target, N):
    """The number of tokens a model with N parameters needs to reach the loss `target` (infinite if its floor
    E + A/N^α is already higher)."""
    gap = target - f["E"] - f["A"] / N ** f["alpha"]
    return (f["B"] / gap) ** (1 / f["beta"]) if gap > 0 else math.inf


def golden(fn, lo, hi, iters=200):
    """Find the minimum of a unimodal function on [lo, hi] by golden-section search."""
    g = (math.sqrt(5) - 1) / 2
    a, b = lo, hi
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = fn(c), fn(d)
    for _ in range(iters):
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - g * (b - a)
            fc = fn(c)
        else:
            a, c, fc = c, d, fd
            d = a + g * (b - a)
            fd = fn(d)
    return (a + b) / 2


def cheapest(f, target, T):
    """The cheapest N, D (training + generating T tokens) that reach the loss target; searches over ln N."""
    lo = math.log((f["A"] / (target - f["E"])) ** (1 / f["alpha"])) + 1e-9  # the smallest N that can still reach it

    def cost(lN):
        return 6 * math.exp(lN) * tokens_for(f, target, math.exp(lN)) + 2 * math.exp(lN) * T

    N = math.exp(golden(cost, lo, lo + 16))
    return N, tokens_for(f, target, N), cost(math.log(N))


fr = CH_FITS["Besiroglu 2024"]
m8 = models["Llama 3 8B"]
target = float(predict(fr, m8["N"], m8["D"]))
C8 = 6 * m8["N"] * m8["D"]
N0, D0, C0 = cheapest(fr, target, 0.0)
lNs = np.linspace(math.log(N0) - 3, math.log(N0) + 3, 600001)  # check by scanning
gap = target - fr["E"] - fr["A"] / np.exp(lNs) ** fr["alpha"]
ok = gap > 0  # if N is too small, the floor E + A/N^α is already above the target
grid_cost = np.full_like(lNs, np.inf)
grid_cost[ok] = 6 * np.exp(lNs[ok]) * (fr["B"] / gap[ok]) ** (1 / fr["beta"])
assert abs(lNs[grid_cost.argmin()] - math.log(N0)) < 1e-4 and grid_cost.min() >= C0 * (1 - 1e-12)
print(
    f"Llama 3 8B: {fmt(m8['N'], 0)} parameters × {fmt(m8['D'] / 1e12, 0)} trillion tokens, "
    f"D/N = {fmt(m8['D'] / m8['N'], 0)} (nearly {fmt(round(m8['D'] / m8['N'], -2), 0)})"
)
print(f"its loss level (according to the refit): {fmt(target, 3)} nat/token")
print(
    f"cheapest to train to that level: {fmt(N0 / 1e9, 1)} billion parameters × {fmt(D0 / 1e9, 0)} billion tokens (D/N = {fmt(D0 / N0, 0)})"
)
print(f"  {fmt(C8 / C0, 1)} times cheaper to train; each generated token {fmt(N0 / m8['N'], 1)} times more expensive")
T_even = (C8 - C0) / (2 * (N0 - m8["N"]))
print(f"the 8B model breaks even after T = {sci(T_even, 2)} generated tokens")
for T in (1e12, 1e13, 1e14):
    N, D, cost = cheapest(fr, target, T)
    print(
        f"  if it will generate {sci(T, 0)} tokens: the cheapest is {fmt(N / 1e9, 1)} billion parameters, D/N = {fmt(D / N, 0)}"
    )
assert abs(C8 / C0 - 5.50) < 0.01 and abs(N0 / m8["N"] - 4.22) < 0.01 and 1.1e13 < T_even < 1.2e13, (
    "differs from the demo"
)

According to the refit, the loss level of Llama 3 8B is reached most cheaply by a model of about 34 billion
parameters trained on about 650 billion tokens, exactly 19 tokens per parameter. Training that model is 5.5 times
cheaper, but each token it generates costs 4.2 times more. After about $1.1 \cdot 10^{13}$ generated tokens, the
“overtrained” 8B model is cheaper, counting both parts. The more you plan to generate, the smaller the cheapest model
and the longer it trains: with $10^{14}$ tokens, the optimal ratio is 1,136 tokens per parameter.

Note: Chinchilla's constants were measured on MassiveText with Gopher's tokenizer, not on Llama 3's data, so the
numbers above only illustrate the calculation. Sardana et al. (2024) actually trained models up to $10^4$ tokens per
parameter and found that quality kept improving. MoE (Lesson 24) is another way to have many parameters while each
token stays cheap.

## 13 · “Emergent” abilities: the model or the metric?

Wei et al. (2022) call an ability “emergent” if it is nearly absent in small models and then appears suddenly in
large ones. Schaeffer et al. (2023) showed that most of those abilities are measured with **all-or-nothing** metrics
such as exact match of the whole string.

The demo measured both kinds of metric on our models ($D = 2^{22}$, 3 seeds per size, 6 sizes including the
held-out size). The smooth metric is bits per character. The all-or-nothing metric is: after a space, does the model
predict (argmax) **every** character of the next $k$ syllables correctly (Vietnamese writes each syllable as a
separate word), i.e. would greedy decoding copy them out exactly? The partial metric is the fraction of correct
characters within those same snippets.

### 🤔 Predict first
The largest model gets about 53% of the characters right in 4-syllable snippets (18.5 characters long on average). If
each character were right independently with probability 0.53, what is the probability that the whole snippet is
right? Out of 324,666 snippets (pooled over 18 models), how many do you guess are exact matches?

In [ ]:
EM_D = [16, 32, 48, 64, 96, 128]
em = {d: [r["em"] for r in grid + held if r["d"] == d and r["D"] == DMAX] for d in EM_D}
assert all(len(v) == 3 for v in em.values())


def avg(d, k):
    return float(np.mean([e[k] for e in em[d]]))


for d in EM_D:
    print(
        f"d {d:>3} ({fmt(size[d]['total'], 0):>7} parameters): {fmt(avg(d, 'bpc'), 3)} bits/character · correct characters {pct(avg(d, 'acc'), 1)} · "
        f"exact match of 1 / 2 syllables {pct(avg(d, 'em1'), 2)} / {pct(avg(d, 'em2'), 3)}, "
        f"of 4 syllables {sum(round(e['em4'] * e['n4']) for e in em[d])} snippets · "
        f"correct characters in 4-syllable snippets {pct(avg(d, 'pc4'), 1)}"
    )
n4, len4 = em[16][0]["n4"], em[16][0]["len4"]
hits4 = sum(round(e["em4"] * e["n4"]) for v in em.values() for e in v)
n_models = sum(len(v) for v in em.values())
p_ind = avg(128, "pc4") ** len4
print(f"4-syllable snippets: {fmt(n4, 0)} per model, {fmt(len4, 1)} characters long on average")
print(f"exact matches: {hits4} out of {fmt(n_models * n4, 0)} snippets ({n_models} models × {fmt(n4, 0)})")
print(
    f"if each character were independent: {fmt(avg(128, 'pc4'), 3)}^{fmt(len4, 1)} = {sci(p_ind, 1)}, i.e. {fmt(p_ind * n4, 2)} out of {fmt(n4, 0)} snippets"
)
who = [f"d {r['d']} seed {r['seed']}" for r in grid + held if r["D"] == DMAX and r["em"]["em4"] > 0]
print("models with an exactly matched 4-syllable snippet:", ", ".join(who))
assert hits4 == 2 and n_models * n4 == 324666, "differs from the demo"

The figure below puts the two metrics side by side against $N$ (log scale). Left: bits per character, the smooth
metric. Right: exact match of 1, 2 and 4 syllables (three colors, three markers), the all-or-nothing metric, along
with the fraction of correct characters in the 4-syllable snippets (dashed line, right axis).

In [ ]:
Ns_em = [size[d]["total"] for d in EM_D]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
a1.plot(Ns_em, [avg(d, "bpc") for d in EM_D], "o-", color=BLUE, ms=6)
a1.set(xscale="log", xlabel="N (parameters, log scale)", ylabel="bits/character", title="Smooth metric: falls steadily")
for k, color, mk in ((1, BLUE, "o"), (2, ORANGE, "s"), (4, GREEN, "^")):
    a2.plot(
        Ns_em,
        [100 * avg(d, f"em{k}") for d in EM_D],
        marker=mk,
        color=color,
        ms=6,
        label=f"exact match, {k} syllable{'s' if k > 1 else ''}",
    )
a2.set(xscale="log", xlabel="N (parameters, log scale)", ylabel="exact match (%)", title="All-or-nothing metric")
a3 = a2.twinx()
a3.plot(
    Ns_em,
    [100 * avg(d, "pc4") for d in EM_D],
    "--",
    color=GREY,
    lw=1.5,
    label="correct characters, 4 syllables (right)",
)
a3.set_ylabel("correct characters (%)", color=GREY)
a3.spines["top"].set_visible(False)
h1, l1 = a2.get_legend_handles_labels()
h2, l2 = a3.get_legend_handles_labels()
a2.legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1, 0.06))
for ax in (a1, a2):
    ax.grid(alpha=0.3, which="both")
plt.tight_layout()
fmt_axes(fig)
plt.show()
steps = [avg(a, "bpc") - avg(b, "bpc") for a, b in itertools.pairwise(EM_D)]
print(
    f"N ×{fmt(NH / size[16]['total'], 0)}: bits/character {fmt(avg(16, 'bpc'), 2)} → {fmt(avg(128, 'bpc'), 2)}; "
    "each step up to the next size takes off " + " · ".join(fmt(x, 3) for x in steps)
)
assert all(x > 0 for x in steps) and all(a > b for a, b in itertools.pairwise(steps))  # falls steadily, no jumps
print(
    f"correct characters ×{fmt(avg(128, 'acc') / avg(16, 'acc'), 2)}, but exact match of 1 syllable ×{fmt(avg(128, 'em1') / avg(16, 'em1'), 1)}"
)

Same models, two different stories:
- The **smooth metric** (bits per character) falls from 2.95 to 2.54 as $N$ grows 61-fold, with no jump anywhere;
  each step up to the next size takes off less than the step before.
- The **all-or-nothing metric** magnifies that same progress: the fraction of correct characters grows only
  1.14-fold, but exact match of 1 syllable grows 5.6-fold. For 4 syllables, an exact match almost never happens:
  2 times in 324,666 snippets, both from a single d 32 model, not the largest model. The independence estimate for
  the largest model is 0.16 snippets out of 18,037, right in that range.

If we kept scaling up, the fraction of correct characters would creep upward, and its 18.5th power would shoot from
“nearly 0” to “substantial” within a few model sizes. Seen through the exact-match metric, that is an “emergent”
ability, even though the model underneath improves steadily. Schaeffer et al. (2023) found that most (over 92%) of
the “emergent” abilities on BIG-Bench are measured with multiple-choice grading or exact string match; switch to a
continuous metric and the curve is usually smooth.

That doesn't mean every ability is predictable, and predicting the **loss** doesn't by itself give a prediction of
**accuracy**. Llama 3 had to add a step: from FLOP to log-likelihood on the benchmark, then from log-likelihood to
accuracy via a sigmoid curve.

## 14 · Try changing things

The cells below rerun the same `fit()` function with different choices. You can edit the values and run them again.

### The Huber threshold $\delta$
Chinchilla used $\delta = 10^{-3}$, so small that almost every point falls in the linear part. Try larger values:

In [ ]:
for delta in (1e-3, 1e-2, 1e-1):
    f = fit(P_all, delta=delta)
    N12, D12, _ = frontier(f, 1e12)
    print(
        f"δ = {fmt(delta, 3)}: E {fmt(f['E'], 3)} · α {fmt(f['alpha'], 3)} · β {fmt(f['beta'], 3)}; "
        f"d 128 at 2²²: {fmt(predict(f, NH, DMAX), 3)}; optimal D/N at 10¹²: {fmt(D12 / N12, 0)}"
    )

The prediction for the held-out model barely changes (by less than 0.01 nat), but the optimal ratio jumps from 165
to 230 at $\delta = 0.1$. Once again, the quantity we *measure* is robust, while the quantity we *extrapolate* is
not.

### Sum or mean?
Besiroglu et al. (2024) argue that the original took the **mean** of the Huber loss instead of the **sum**.
Mathematically, the two objectives have the same minimum. But the mean is 75 times smaller, and so is its gradient,
so an optimizer that stops once the gradient is small enough stops earlier. The cell below lets BFGS stop when
$|g| < 10^{-5}$ (instead of $10^{-13}$) for both objectives.

In [ ]:
res_ms = {}
for name, scale in (("sum", 1.0), ("mean", 1 / len(P_all))):
    f = fit(P_all, scale=scale, gtol=1e-5)
    res_ms[name] = f
    print(
        f"{name:<4}: α {fmt(f['alpha'], 3)} · β {fmt(f['beta'], 3)} · E {fmt(f['E'], 3)}; Huber sum {fmt(f['obj'] * 1e4, 3)}·10⁻⁴"
    )
assert res_ms["mean"]["obj"] > 1.1 * res_ms["sum"]["obj"]  # the mean stops at a clearly worse point

With the sum, BFGS still reaches the minimum of section 5. With the mean, it stops at a point with a clearly higher
objective and very different exponents. Same data, same formula, only a constant factor in the objective. This is
one of the reasons Besiroglu et al. give for why the constants printed in the paper deviate from approaches 1 and 2.

### Resampling the runs (bootstrap)
How certain are the constants? We draw 75 runs with replacement from the 75 runs, refit, and repeat a few times.
The demo did this 200 times; here we do it 12 times to keep it fast.

In [ ]:
boot_rng = np.random.default_rng(5)
boot = []
for _ in range(12):
    f = fit(P_all[boot_rng.integers(0, len(P_all), len(P_all))])
    N12, D12, _ = frontier(f, 1e12)
    boot.append((f["alpha"], f["beta"], f["beta"] / (f["alpha"] + f["beta"]), D12 / N12))
boot = np.array(boot)
for j, name in enumerate(("α", "β", "a = β/(α+β)", "optimal D/N at 10¹²")):
    dg = 0 if j == 3 else 2
    print(f"{name:<19}: {fmt(boot[:, j].min(), dg)}–{fmt(boot[:, j].max(), dg)} over 12 resamples")
A_CI_CHINCHILLA = (0.454, 0.455)  # Hoffmann et al. 2022, table 2: the 10–90% interval of a, approach 3
print(
    f"Chinchilla, approach 3: a from {fmt(A_CI_CHINCHILLA[0], 3)} to {fmt(A_CI_CHINCHILLA[1], 3)}; "
    f"our range is ~{fmt(round(np.ptp(boot[:, 2]) / np.ptp(A_CI_CHINCHILLA), -1), 0)} times wider"
)
b200 = FITS["boot"]["total"]
print(
    f"demo, 200 resamples (10–90%): α {fmt(b200['alpha'][0], 2)}–{fmt(b200['alpha'][2], 2)} · "
    f"D/N at 10¹² {fmt(b200['ratio']['1e+12'][0], 0)}–{fmt(b200['ratio']['1e+12'][2], 0)}"
)

Just changing *which runs* are used already makes $\alpha$ swing by about ±0.1, and the optimal ratio at
$C = 10^{12}$ can be 130 or 210 characters per parameter. The 10–90% interval that Hoffmann et al. report for $a$ in
approach 3 is about 100 times narrower. Besiroglu et al. argue that an interval that narrow would take more than
$6 \cdot 10^5$ runs, while Chinchilla trained over 400 models.

### Only one seed
Each cell of the grid has 3 seeds. Try fitting only seed 0 (25 runs) and compare with 3 seeds:

In [ ]:
P_seed0 = points([r for r in grid if r["seed"] == 0])
f_s0 = fit(P_seed0)
print(
    f"seed 0 only: E {fmt(f_s0['E'], 3)} · α {fmt(f_s0['alpha'], 3)} · β {fmt(f_s0['beta'], 3)}; d 128 at 2²²: {fmt(predict(f_s0, NH, DMAX), 3)}"
)
print(
    f"all 3 seeds: E {fmt(f0['E'], 3)} · α {fmt(f0['alpha'], 3)} · β {fmt(f0['beta'], 3)}; d 128 at 2²²: {fmt(predict(f0, NH, DMAX), 3)}"
)
assert abs(f_s0["alpha"] - FITS["fits"]["seed0-total-sched"]["alpha"]) < 2e-3, "differs from the demo"

One seed (25 runs) gives exponents slightly different from 3 seeds, but the prediction for the held-out model
differs by only 0.001 nat.

## 15 · Limits, and next steps

What this notebook *cannot* tell us:
- **The measured range is very small.** Our models have 13,952–850,880 parameters. Chinchilla's smallest model, at
  70 million, is already 82 times as large as the largest one here. Every fit is only trustworthy within the measured
  range, and the further we extrapolate, the more the result depends on the exponent digits that we have just seen
  are uncertain.
- **The data is unlimited.** Every run sees each character exactly once. When data is limited and has to be repeated
  over many epochs, each repeated token is worth less and less (Muennighoff et al., 2023), and the formula needs an
  extra term.
- **Loss is not capability.** As in section 13, getting from the loss to accuracy on a task takes an extra step.
- **One kind of architecture.** The constants change with the architecture. Lesson 24 changes the very calculation
  $C \approx 6ND$: with Mixture of Experts, each token goes through only a few “experts”, so the number of parameters
  and the computation per token come apart.

In [ ]:
rng_ch = CHIN["range"]  # Hoffmann et al. 2022: "70 million to over 16 billion parameters on 5 to 500 billion tokens"
print(
    f"here: {fmt(size[16]['total'], 0)}–{fmt(NH, 0)} parameters, {fmt(DS[0] / 1e6, 2)}–{fmt(DMAX / 1e6, 2)} million characters"
)
print(
    f"Chinchilla: {fmt(rng_ch['Nmin'] / 1e6, 0)} million – {fmt(rng_ch['Nmax'] / 1e9, 0)} billion parameters; their smallest model "
    f"is {fmt(rng_ch['Nmin'] / NH, 0)} times as large as the largest one here"
)

## Summary
- Training costs $C \approx 6ND$: each weight does 1 multiply-accumulate in the forward pass and 2 in the backward
  pass. GPT-2 small (124,439,808 parameters) comes to 797,870,592 FLOP per token when every multiplication is
  counted, 1.07 times $6N$; attention accounts for 7%.
- Chinchilla's form, $L = E + A/N^\alpha + B/D^\beta$, fitted with a Huber loss on $\ln L$, sums up 75 runs with
  5 constants (off by 0.035 nat). The reducible part is a power law; $L$ itself bends toward a floor (1.83 for the
  smallest model).
- The fit predicts an unseen model 1.7 times larger with an error of 0.017 nat. Predictions are more robust than
  exponents: four subsets give $\beta$ from 0.47 to 0.77, but predictions only 0.024 apart.
- Compute-optimal has a closed form, $N_\text{opt} \propto C^{\beta/(\alpha+\beta)}$. Our optimal ratio: 85–115
  characters per parameter from the IsoFLOP valleys, 140–210 from the fit, and the two methods even move in opposite
  directions.
- “20 tokens per parameter” is not a law: the constants printed in the Chinchilla paper give 93, the refit gives 18,
  Llama 3 gives 41.
- Llama 3 8B was trained on nearly 1,900 tokens per parameter because it will generate a great deal: it is cheaper
  than the compute-optimal model after about $10^{13}$ tokens.
- “Emergence” is mostly due to the metric: bits per character fall steadily, while exact match of 4 syllables
  happens 2 times in 324,666.

## Review questions
1. Why does one weight cost 6 FLOP per training token, and why does the embedding table cost almost no FLOP?
2. On a log–log scale, the smallest model's loss as a function of $D$ is curved, not straight. Why? Which quantity is
   straight?
3. With $C = 6ND$ fixed, prove that if $\alpha = \beta$ then the optimal ratio $D/N$ does not depend on $C$.
4. The constants printed in the Chinchilla paper give about 93 tokens per parameter, yet Chinchilla itself was
   trained on 20. Give two reasons why 20 is still considered reasonable.
5. The optimal ratio of the character-level models here is about 100 characters per parameter. Does that contradict
   “20 tokens per parameter”?
6. A team plans to serve $10^{14}$ tokens with its model. Should it choose a model smaller or larger than the
   compute-optimal one, and train it on more or fewer tokens?
7. An ability measured by exact match “appears suddenly” at some model size. What would you check before calling it
   emergent?

<details><summary>Hints and answers</summary>

1. Forward pass: 1 MAC ($xW$); backward pass: 2 MACs (the gradients with respect to the input and to the weights).
   Each MAC is 2 FLOP. The embedding table is only looked up (one row is selected), not multiplied as a matrix;
   unless it is also the output layer, as in GPT-2.
2. Because the loss has a floor $E + A/N^\alpha$ that no amount of extra data gets past. It is the part
   $L - E - A/N^\alpha = B/D^\beta$ that is straight, with slope $-\beta$.
3. $N_\text{opt} \propto C^{\beta/(\alpha+\beta)} = C^{1/2}$ and $D_\text{opt} \propto C^{1/2}$, so $D/N = G^{-2}$, a
   constant.
4. The paper's own approaches 1 and 2 give $a \approx 0.5$ and a ratio of about 20; the refit (Besiroglu et al., 2024)
   gives 18, and shows that the original stopped its optimization too early. The IsoFLOP curves are also flat near
   the bottom, so being off by a factor of a few around the optimum costs little.
5. No: one counts characters, the other BPE tokens a few characters long; the ratio also depends on the data, the
   architecture and the estimation method (here the two methods give 85–115 and 140–210).
6. Smaller, and trained on more tokens: each generated token costs about $2N$ FLOP, so for large $T$ the generation
   outweighs the training (section 12: with $10^{14}$ tokens, about 8.8 billion parameters and 1,136 tokens per
   parameter).
7. Measure the same models with a continuous metric (log-likelihood, the fraction of correct characters or tokens).
   If it rises steadily, the jump comes from the all-or-nothing metric, not from the model.
</details>

## Further reading
- J. Kaplan et al. (2020), [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361).
- J. Hoffmann et al. (2022), [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556)
  (Chinchilla): approaches 1–3, appendix D.2 (the fit), table A4 ($6ND$).
- T. Besiroglu et al. (2024), [Chinchilla Scaling: A replication attempt](https://arxiv.org/abs/2404.10102), and
  [Epoch AI's blog post](https://epoch.ai/blog/chinchilla-scaling-a-replication-attempt).
- T. Pearce & J. Song (2024), [Reconciling Kaplan and Chinchilla Scaling Laws](https://arxiv.org/abs/2406.12907):
  how the embedding parameters are counted.
- T. Porian et al. (2024), [Resolving Discrepancies in Compute-Optimal Scaling of Language Models](https://arxiv.org/abs/2406.19146).
- N. Sardana et al. (2024), [Beyond Chinchilla-Optimal: Accounting for Inference](https://arxiv.org/abs/2401.00448).
- Llama Team (2024), [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783), section 3.2.1: its own scaling
  law and the accuracy-prediction step.
- N. Muennighoff et al. (2023), [Scaling Data-Constrained Language Models](https://arxiv.org/abs/2305.16264).
- J. Wei et al. (2022), [Emergent Abilities of Large Language Models](https://arxiv.org/abs/2206.07682); R. Schaeffer,
  B. Miranda & S. Koyejo (2023), [Are Emergent Abilities of Large Language Models a Mirage?](https://arxiv.org/abs/2304.15004)
- Epoch AI, [Training tokens per parameter of open-weight models](https://epoch.ai/data-insights/training-tokens-per-parameter).
- Stanford CS336 (2025), [Lecture 9: Scaling laws basics](https://raw.githubusercontent.com/stanford-cs336/spring2025-lectures/main/nonexecutable/2025%20Lecture%209%20-%20Scaling%20laws%20basics.pdf);
  L. Weng (2026), [Scaling Laws, Carefully](https://lilianweng.github.io/posts/2026-06-24-scaling-laws/).

**Next lesson:** [Lesson 24 · Mixture of Experts (MoE)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mixture-of-experts/):
many parameters while each token stays cheap, when $C \approx 6ND$ counts only the experts a token actually goes
through.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")